# 🎓 NexaPulse – Autonomous Predictive Action Agent
### Use-case: Student Performance

**Concept:** Students-oda past + current data-va analyze panni → yaaru final exam-la fail aagalaam nu **munnaadiye predict** panni → avangalukku suitable actions-a (alert, extra practice, mentoring, guardian notice) **automatically plan & execute** pannum AI Agent.

**Agent loop:** `Perceive → Predict → Diagnose → Plan → Act → Learn`

> Ella data-vum synthetic, so dataset download vendaam. `Kernel → Restart & Run All` pannunga.

## 1. Setup & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, classification_report, roc_auc_score

import warnings
warnings.filterwarnings("ignore")

CONFIG = {
    "n_past": 400,            # last semester students (training data)
    "n_live": 120,            # current semester students (agent monitors them)
    "weeks": 14,              # semester length
    "start_week": 4,          # agent starts predicting from this week
    "pass_mark": 40,          # final score below this = FAIL
    "risk_threshold": 0.50,   # agent acts when risk >= this (agent will self-adjust)
    "cooldown_weeks": 4,      # same action not repeated within these weeks (guardian notice = once only)
    "seed": 42,
    "simulation_mode": True,  # True = messages/actions only simulated (safe)
    # ASSUMPTION for simulation: how many final-score points each action can add
    "intervention_effect": {
        "attendance_alert": 1.5, "assign_extra_practice": 2.5, "deadline_reminders": 1.0,
        "study_plan": 2.0, "schedule_mentoring": 4.0, "notify_guardian": 2.0, "teacher_alert": 0.5,
    },
    "max_boost": 12,          # max points an intervention package can add to one student
}
plt.rcParams["figure.figsize"] = (11, 3.5)
print("NexaPulse (Student Performance) ready ✅")

## 2. Data Layer – Past semester + Current semester

In [ ]:
def generate_students(n, weeks, seed, prefix):
    rng = np.random.default_rng(seed)
    ability = rng.normal(0, 1, n)
    motivation = rng.normal(0, 1, n)
    disengaged = rng.random(n) < 0.22                 # students who slowly lose interest
    drop_start = rng.integers(3, 8, n)
    w = np.arange(1, weeks + 1)
    rows, finals = [], []

    for i in range(n):
        decay = np.where(disengaged[i], np.clip(w - drop_start[i], 0, None), 0)
        att = np.clip(82 + 7 * motivation[i] + 2 * ability[i] - 4.0 * decay + rng.normal(0, 5, weeks), 20, 100)
        study = np.clip(8 + 2.5 * motivation[i] + ability[i] - 0.7 * decay + rng.normal(0, 1.5, weeks), 0, 30)
        quiz = np.clip(55 + 11 * ability[i] + 3 * motivation[i] + 0.25 * (att - 75) + 1.2 * (study - 8)
                       + rng.normal(0, 7, weeks), 0, 100)
        assign = np.clip(58 + 10 * ability[i] + 4 * motivation[i] + 0.2 * (att - 75) + rng.normal(0, 7, weeks), 0, 100)
        late = rng.poisson(np.clip(1.0 - 0.5 * motivation[i] + 0.5 * decay, 0.05, 5))
        sid = f"{prefix}{i + 1:03d}"
        for k in range(weeks):
            rows.append((sid, k + 1, att[k], study[k], quiz[k], assign[k], late[k]))
        final = (53 + 12 * ability[i] + 5 * motivation[i] + 0.3 * (att.mean() - 75)
                 + 1.0 * (study.mean() - 8) + rng.normal(0, 5))
        finals.append((sid, float(np.clip(final, 0, 100))))

    rec = pd.DataFrame(rows, columns=["student_id", "week", "attendance", "study_hours", "quiz", "assignment", "late_count"])
    fin = pd.DataFrame(finals, columns=["student_id", "final_score"]).set_index("student_id")
    return rec, fin

past_rec, past_final = generate_students(CONFIG["n_past"], CONFIG["weeks"], CONFIG["seed"], "P")
live_rec, live_final = generate_students(CONFIG["n_live"], CONFIG["weeks"], CONFIG["seed"] + 1, "S")

fail_rate = (past_final.final_score < CONFIG["pass_mark"]).mean()
print("Past records:", past_rec.shape, "| Live records:", live_rec.shape)
print(f"Past semester fail rate: {fail_rate:.1%}")
past_rec.head(8)

## 3. Analysis – Past data-la enna pattern irukku?

In [ ]:
past = past_rec.merge(past_final, left_on="student_id", right_index=True)
past["result"] = np.where(past.final_score < CONFIG["pass_mark"], "FAIL", "PASS")

weekly = past.groupby(["week", "result"])[["attendance", "quiz", "study_hours"]].mean().reset_index()
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for j, col in enumerate(["attendance", "quiz", "study_hours"]):
    for res, colr in [("PASS", "tab:green"), ("FAIL", "tab:red")]:
        d = weekly[weekly.result == res]
        ax[j].plot(d.week, d[col], label=res, color=colr, marker="o", ms=3)
    ax[j].set_title(f"Weekly average {col}"); ax[j].set_xlabel("week")
ax[0].legend(); plt.tight_layout(); plt.show()

per_student = past.groupby("student_id").agg(att=("attendance", "mean"), quiz=("quiz", "mean"), final=("final_score", "first"))
print("Correlation with final score:")
print(per_student.corr()["final"].round(2))

## 4. Prediction Engine – yaaru fail aaga chance irukku?

In [ ]:
def build_features(g, t):
    g = g[g.week <= t].sort_values("week")
    l3, f3 = g.tail(3), g.head(3)
    return {
        "week": t,
        "att_mean": g.attendance.mean(), "att_recent": l3.attendance.mean(), "att_trend": l3.attendance.mean() - f3.attendance.mean(),
        "quiz_mean": g.quiz.mean(), "quiz_recent": l3.quiz.mean(), "quiz_trend": l3.quiz.mean() - f3.quiz.mean(),
        "assign_mean": g.assignment.mean(),
        "late_total": g.late_count.sum(),
        "study_mean": g.study_hours.mean(), "study_recent": l3.study_hours.mean(),
    }

def snapshot_table(records, t):
    # features of every student using data up to week t
    return pd.DataFrame({sid: build_features(g, t) for sid, g in records.groupby("student_id")}).T.astype(float)

# training set: every student x every week (start_week .. weeks-1)
parts = []
for t in range(CONFIG["start_week"], CONFIG["weeks"]):
    snap = snapshot_table(past_rec, t)
    snap["final"] = past_final.loc[snap.index, "final_score"]
    parts.append(snap)
data = pd.concat(parts)
data["fail"] = (data.final < CONFIG["pass_mark"]).astype(int)

FEATURES = [c for c in data.columns if c not in ("final", "fail")]
ids = past_final.index.tolist()
train_ids, test_ids = ids[:300], ids[300:]            # split by STUDENT (no leakage)
tr, te = data[data.index.isin(train_ids)], data[data.index.isin(test_ids)]

clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=1, n_jobs=-1)
reg = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=1, n_jobs=-1)
clf.fit(tr[FEATURES], tr.fail); reg.fit(tr[FEATURES], tr.final)

print("Final-score prediction MAE :", round(mean_absolute_error(te.final, reg.predict(te[FEATURES])), 2), "marks")
print("Fail-risk ROC-AUC          :", round(roc_auc_score(te.fail, clf.predict_proba(te[FEATURES])[:, 1]), 3))
print()
print(classification_report(te.fail, clf.predict(te[FEATURES]), target_names=["pass", "fail"], zero_division=0))

early = te[te.week == 6]
plt.scatter(early.final, reg.predict(early[FEATURES]), s=12, alpha=0.6)
plt.plot([0, 100], [0, 100], "r--"); plt.axvline(CONFIG["pass_mark"], color="gray", ls=":")
plt.xlabel("Actual final score"); plt.ylabel("Predicted at week 6"); plt.title("Week-6 prediction vs actual final (unseen students)")
plt.show()

## 5. The Agent – Perceive → Predict → Diagnose → Plan → Act → Learn

- **Perceive:** indha week varaikkum student data-va eduthukkum
- **Predict:** fail risk + expected final score
- **Diagnose:** *en* risk? (low attendance, falling quiz, late submissions, low study time)
- **Plan:** reason + risk level base panni actions choose pannum
- **Act:** student/teacher/guardian-ku messages & tasks (simulation mode-la safe)
- **Learn:** semester mudinjadhum flags correct-ah nu check panni threshold-a self-adjust pannum

In [ ]:
MESSAGES = {
    "attendance_alert":      "Hi {sid}, your recent attendance is {att:.0f}%. Please attend the upcoming classes - they matter for your exam.",
    "assign_extra_practice": "Hi {sid}, your recent quiz average is {quiz:.0f}. We have assigned extra practice sets for you.",
    "deadline_reminders":    "Hi {sid}, you have {late:.0f} late submissions. Turn on deadline reminders - we will nudge you 2 days before.",
    "study_plan":            "Hi {sid}, you study about {study:.1f} hrs/week. Here is a simple weekly study plan to get back on track.",
    "schedule_mentoring":    "Hi {sid}, a 1-to-1 mentoring session has been scheduled for you this week.",
    "notify_guardian":       "Dear Guardian, {sid} may need extra support this semester. Please connect with the class mentor.",
    "teacher_alert":         "[TEACHER] {sid} flagged: risk {risk:.0%}, predicted final {pred:.0f}. Please review.",
}

class StudentSuccessAgent:
    def __init__(self, reg, clf, config):
        self.reg, self.clf, self.cfg = reg, clf, config
        self.threshold = config["risk_threshold"]
        self.last_done = {}          # (student, action) -> week (cooldown memory)
        self.boost = {}              # student -> total simulated improvement
        self.flagged = set()
        self.log, self.risk_rows, self.outbox = [], [], []

    # 1. PERCEIVE
    def perceive(self, records, t):
        return snapshot_table(records, t)[FEATURES]

    # 2. PREDICT
    def predict(self, feats):
        return pd.DataFrame({"risk": self.clf.predict_proba(feats)[:, 1],
                             "pred_final": self.reg.predict(feats)}, index=feats.index)

    # 3. DIAGNOSE
    def diagnose(self, r):
        reasons = []
        if r.att_recent < 70 or r.att_trend < -10: reasons.append("low attendance")
        if r.quiz_recent < 45 or r.quiz_trend < -8: reasons.append("falling quiz scores")
        if r.late_total >= 6: reasons.append("late submissions")
        if r.study_recent < 5: reasons.append("low study time")
        return reasons

    # 4. PLAN
    def plan(self, reasons, risk):
        if risk < self.threshold:
            return []
        m = {"low attendance": "attendance_alert", "falling quiz scores": "assign_extra_practice",
             "late submissions": "deadline_reminders", "low study time": "study_plan"}
        acts = [m[x] for x in reasons]
        if risk >= 0.75: acts += ["schedule_mentoring", "teacher_alert"]
        if risk >= 0.90: acts.append("notify_guardian")
        return acts or ["teacher_alert"]

    # 5. ACT
    def act(self, sid, t, acts, row, risk, pred):
        done = []
        for a in acts:
            wait = 99 if a == "notify_guardian" else self.cfg["cooldown_weeks"]
            if t - self.last_done.get((sid, a), -99) < wait:
                continue                                           # don't spam the same student
            self.last_done[(sid, a)] = t
            self.boost[sid] = self.boost.get(sid, 0) + self.cfg["intervention_effect"][a]
            msg = MESSAGES[a].format(sid=sid, att=row.att_recent, quiz=row.quiz_recent, late=row.late_total,
                                     study=row.study_recent, risk=risk, pred=pred)
            self.outbox.append({"week": t, "student_id": sid, "action": a, "message": msg,
                                "mode": "SIMULATED" if self.cfg["simulation_mode"] else "LIVE"})
            done.append(a)
        return done

    # ONE WEEK CYCLE for all students
    def run_week(self, records, t):
        feats = self.perceive(records[records.week <= t], t)
        pred = self.predict(feats)
        for sid in feats.index:
            risk, pf = pred.loc[sid, "risk"], pred.loc[sid, "pred_final"]
            reasons = self.diagnose(feats.loc[sid])
            acts = self.plan(reasons, risk)
            done = self.act(sid, t, acts, feats.loc[sid], risk, pf)
            if acts: self.flagged.add(sid)
            self.risk_rows.append({"week": t, "student_id": sid, "risk": risk, "pred_final": pf})
            if done:
                self.log.append({"week": t, "student_id": sid, "risk": round(risk, 2), "pred_final": round(pf, 1),
                                 "reasons": reasons, "actions": done})

    # 6. LEARN (end of semester)
    def learn(self, truly_failing):
        flagged = self.flagged
        tp = len(flagged & truly_failing)
        precision = tp / max(len(flagged), 1)
        recall = tp / max(len(truly_failing), 1)
        old = self.threshold
        if precision < 0.5:  self.threshold = min(0.85, self.threshold + 0.05)   # too many false alarms
        if recall < 0.9:     self.threshold = max(0.20, self.threshold - 0.05)   # missed some failing students
        return precision, recall, old, self.threshold

agent = StudentSuccessAgent(reg, clf, CONFIG)
print("StudentSuccessAgent created ✅")

## 6. Run the Agent on CURRENT semester (week by week)

Agent week 4-la irundhu week 13 varaikkum ovvoru weeka students-a monitor pannum.
> **Simulation assumption:** action edutha students-oda final score konjam improve aagum (`intervention_effect` config-la irukku, neenga maathalaam). Ithu real-world proof illa, demo-kaaga mattum.

In [ ]:
for t in range(CONFIG["start_week"], CONFIG["weeks"]):
    agent.run_week(live_rec, t)

risk_hist = pd.DataFrame(agent.risk_rows)
outbox = pd.DataFrame(agent.outbox)
log = pd.DataFrame(agent.log)

# simulated outcome (students respond differently to help)
rng = np.random.default_rng(7)
responsiveness = pd.Series(rng.uniform(0.5, 1.2, len(live_final)), index=live_final.index)
boost = pd.Series(agent.boost).reindex(live_final.index).fillna(0).clip(upper=CONFIG["max_boost"])
final_without = live_final.final_score
final_with = (final_without + boost * responsiveness).clip(upper=100)

print(f"Agent sent {len(outbox)} actions to {outbox.student_id.nunique()} students over {CONFIG['weeks'] - CONFIG['start_week']} weeks")
log.head(8)

## 7. Results Dashboard

In [ ]:
pm = CONFIG["pass_mark"]
fail_wo = set(final_without[final_without < pm].index)
fail_w = set(final_with[final_with < pm].index)

print("=" * 56)
print(" NEXAPULSE RESULT SUMMARY - current semester")
print("=" * 56)
print(f" Students monitored            : {len(live_final)}")
print(f" Students who would FAIL (no agent): {len(fail_wo)}")
print(f" Students FAIL (with agent)        : {len(fail_w)}")
if fail_wo:
    print(f" Students saved                    : {len(fail_wo - fail_w)}  ({len(fail_wo - fail_w) / len(fail_wo):.0%})")
print(f" Average final score               : {final_without.mean():.1f} -> {final_with.mean():.1f}")

# early-warning quality at week 6
w6 = risk_hist[risk_hist.week == 6].set_index("student_id")
flag6 = set(w6[w6.risk >= CONFIG["risk_threshold"]].index)
print()
print(f" EARLY WARNING at week 6: flagged {len(flag6)} students")
print(f"   caught {len(flag6 & fail_wo)} of {len(fail_wo)} students who would fail (recall {len(flag6 & fail_wo) / max(len(fail_wo),1):.0%})")
print(f"   precision {len(flag6 & fail_wo) / max(len(flag6),1):.0%}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
# (1) risk curves
top = list(fail_wo)[:4] + [s for s in live_final.index if s not in fail_wo][:2]
for sid in top:
    d = risk_hist[risk_hist.student_id == sid]
    ax[0].plot(d.week, d.risk, marker="o", ms=3, label=f"{sid} ({'FAIL' if sid in fail_wo else 'pass'})",
               ls="-" if sid in fail_wo else "--")
ax[0].axhline(CONFIG["risk_threshold"], color="red", ls=":"); ax[0].set_title("Fail-risk over weeks"); ax[0].set_xlabel("week"); ax[0].legend(fontsize=7)
# (2) actions
outbox.action.value_counts().sort_values().plot(kind="barh", ax=ax[1], color="tab:purple"); ax[1].set_title("Actions executed")
# (3) score distribution
ax[2].hist(final_without, bins=20, alpha=0.5, label="without agent", color="tab:red")
ax[2].hist(final_with, bins=20, alpha=0.5, label="with agent", color="tab:green")
ax[2].axvline(pm, color="black", ls="--"); ax[2].set_title("Final score distribution"); ax[2].legend()
plt.tight_layout(); plt.show()

## 8. Teacher Report & Explainability – "Yaaru? Yen? Enna action?"

In [ ]:
last = risk_hist[risk_hist.week == CONFIG["weeks"] - 1].set_index("student_id")
feats_last = snapshot_table(live_rec, CONFIG["weeks"] - 1)

report = last.sort_values("risk", ascending=False).head(10).copy()
report["reasons"] = [", ".join(agent.diagnose(feats_last.loc[s])) or "-" for s in report.index]
report["actions_taken"] = [int((outbox.student_id == s).sum()) for s in report.index]
report["risk"] = report.risk.round(2); report["pred_final"] = report.pred_final.round(1)
print("TOP 10 students needing attention:")
display(report[["risk", "pred_final", "reasons", "actions_taken"]])

sid = report.index[0]
print(f"\n--- Messages sent to {sid} ---")
for _, m in outbox[outbox.student_id == sid].head(4).iterrows():
    print(f"[week {m.week}] ({m['mode']}) {m.message}")

imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values().tail(8)
imp.plot(kind="barh", color="tab:cyan", title="Which signals predict failure the most?"); plt.show()

## 9. Learn – next semester-ku agent-a improve pannu

In [ ]:
precision, recall, old, new = agent.learn(fail_wo)
print(f"Flag precision : {precision:.0%}  (flagged students who really would fail)")
print(f"Flag recall    : {recall:.0%}  (failing students agent caught)")
print(f"Risk threshold : {old:.2f} -> {new:.2f}  (used from next semester)")

## 10. Next Steps

- Real data: `generate_students()` badhula college CSV / Excel / database use pannunga (columns same-ah irukkanum)
- Real actions: email (`smtplib`), WhatsApp/SMS API, Google Calendar for mentoring slots
- Streamlit / Flask dashboard - teacher login pannina risk list paakkalaam (unga web-app skills-ku nalla fit)
- Better models: XGBoost, SHAP explanations
- **Ethics:** student data privacy, bias check, final decision epavum teacher/human kitta irukkanum

**NexaPulse – built by Mukesh P**